In [7]:
import math
import random
import numpy as np
import copy
import matplotlib.pyplot as plt
import time

In [8]:
sdk = '570060003030005060601007000053000001000080000900000270000800402080100030200040019'
def string_to_board(s):
    board = []
    for i in range(0,81,9):
        row = [int(x) for x in s[i:i+9]]
        board.append(row)
    return board


board = string_to_board(sdk)



In [9]:
 ## Sudoku generator and solver - version 1 ##
def draw_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-" * 21)

        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "

            val = board[r][c] if board[r][c] != 0 else "."
            row += str(val) + " "

        print(row)

In [10]:
# # Easy board
# board = [
# [5,3,0,0,7,0,0,0,0],
# [6,0,0,1,9,5,0,0,0],
# [0,9,8,0,0,0,0,6,0],
# [8,0,0,0,6,0,0,0,3],
# [4,0,0,8,0,3,0,0,1],
# [7,0,0,0,2,0,0,0,6],
# [0,6,0,0,0,0,2,8,0],
# [0,0,0,4,1,9,0,0,5],
# [0,0,0,0,8,0,0,7,9]
# ]
#
# # Medium board
# board = [
#     [8,0,0,0,0,0,0,0,0],
#     [0,0,3,6,0,0,0,0,0],
#     [0,7,0,0,9,0,2,0,0],
#     [0,5,0,0,0,7,0,0,0],
#     [0,0,0,0,4,5,7,0,0],
#     [0,0,0,1,0,0,0,3,0],
#     [0,0,1,0,0,0,0,6,8],
#     [0,0,8,5,0,0,0,1,0],
#     [0,9,0,0,0,0,4,0,0]
# ]
# board = [
#     [8,0,0,0,0,0,0,0,9],
#     [0,0,3,0,0,2,1,0,0],
#     [0,7,0,0,9,0,0,0,0],
#     [0,5,0,0,0,7,0,0,0],
#     [0,0,0,8,0,0,7,0,0],
#     [0,0,0,1,0,0,0,3,0],
#     [0,0,1,0,0,0,0,6,0],
#     [0,0,8,0,0,0,0,1,0],
#     [0,9,0,0,0,0,4,0,0]
# ]

draw_board(board)

5 7 . | . 6 . | . . 3 
. 3 . | . . 5 | . 6 . 
6 . 1 | . . 7 | . . . 
---------------------
. 5 3 | . . . | . . 1 
. . . | . 8 . | . . . 
9 . . | . . . | 2 7 . 
---------------------
. . . | 8 . . | 4 . 2 
. 8 . | 1 . . | . 3 . 
2 . . | . 4 . | . 1 9 


In [11]:
def get_block_index(r,c):
    row = (r//3)*3
    column = (c//3)*3
    block_index = row,column
    return block_index

def get_block_cells(block_index):
    block_cells_coordinates = []
    r_start, c_start = block_index[0], block_index[1]

    for i in range(r_start, r_start+3):
        for j in range(c_start, c_start+3):
            block_cells_coordinates.append((i,j))

    return block_cells_coordinates

def find_fixed_cells(board):
    fixed_coordinates = []
    for r in range(9):
         for c in range(9):
             if board[r][c] != 0:
                 fixed_coordinates.append((r,c))

    return fixed_coordinates


def get_block_candidates(board, block_index):
    digits = set(i for i in range(1, 10))
    used_digits = set()
    row_start, col_start = block_index[0], block_index[1]

    for r in range(row_start, row_start+3):
        for c in range(col_start, col_start+3):
            used_digits.add(board[r][c])

    used_digits.discard(0)
    candidates = digits - used_digits
    return candidates

def fill_block(board, block_index, fixed_coordinates):
    row_start, col_start = block_index[0], block_index[1]
    candidates = list(get_block_candidates(board, block_index))
    random.shuffle(candidates)

    for r in range(row_start, row_start+3):
        for c in range(col_start, col_start+3):
            if (r,c) not in fixed_coordinates:
                board[r][c] = candidates.pop()

def initialize_complete_board(board, fixed_coordinates):
    for i in range(0,9,3):
        for j in range(0,9,3):
            fill_block(board, (i,j), fixed_coordinates)


def get_swappable_cells_block(fixed_coordinates):
    swappable_cells = {}

    for r in range(0,9,3):
            for c in range(0,9,3):
                block = (r,c)
                cells = []

                for i in range(r,r+3):
                    for j in range(c,c+3):
                        if (i,j) not in fixed_coordinates:
                            cells.append((i,j))

                swappable_cells[block] = cells

    return swappable_cells

def count_row_conflicts(board, row):
    row_digits = set()

    for cell in range(9):
        row_digits.add(board[row][cell])

    row_conflicts = len(row_digits)
    return row_conflicts

def count_col_conflicts(board, col):
    col_digits = set()

    for cell in range(9):
        col_digits.add(board[cell][col])

    col_conflicts = len(col_digits)
    return col_conflicts

def count_block_conflicts(board, row, col):
    block_digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            block_digits.add(board[r][c])
    return len(block_digits)

def total_energy(board):
    energy = 243

    for row in range(9):
        energy -= count_row_conflicts(board, row)

    for col in range(9):
        energy -= count_col_conflicts(board, col)

    for r in range(0,9,3):
        for c in range(0,9,3):
            energy -= count_block_conflicts(board, r, c)

    return energy

def affected_lines(cell1, cell2):
    affected_rows = list({cell1[0], cell2[0]})
    affected_cols = list({cell1[1], cell2[1]})

    return affected_rows, affected_cols

def local_energy(board, affected_rows, affected_cols):
    energy = 0

    for row in affected_rows:
        energy += count_row_conflicts(board, row)

    for col in affected_cols:
        energy += count_col_conflicts(board, col)

    return energy


def choose_random_cells_random_block(swappable_cells):
    valid_blocks = [block for block, cells in swappable_cells.items() if len(cells) >= 2]
    random_block = random.choice(valid_blocks)

    random_cell_1, random_cell_2 = random.sample(swappable_cells[random_block], 2)

    return random_cell_1, random_cell_2

def swap_cells(board, cell1, cell2):
    r1, c1 = cell1
    r2, c2 = cell2

    board[r1][c1], board[r2][c2] = board[r2][c2] ,board[r1][c1]

def delta_energy_after_swap(board, cell1, cell2):
    affected_rows, affected_cols = affected_lines(cell1, cell2)

    energy_before = total_energy(board)

    swap_cells(board, cell1, cell2)

    energy_after = total_energy(board)

    delta_e = energy_after - energy_before

    return delta_e

def accept_move(delta_e, temp):
    if delta_e <= 0:
        return True
    else:
        p = math.exp(-delta_e/temp)
        r = random.uniform(0,1)
        if r < p:
            return True
        else:
            return False

def update_temp(temp, cooling_rate = 0.999):
    new_temp = temp * cooling_rate
    return new_temp

draw_board(board)


5 7 . | . 6 . | . . 3 
. 3 . | . . 5 | . 6 . 
6 . 1 | . . 7 | . . . 
---------------------
. 5 3 | . . . | . . 1 
. . . | . 8 . | . . . 
9 . . | . . . | 2 7 . 
---------------------
. . . | 8 . . | 4 . 2 
. 8 . | 1 . . | . 3 . 
2 . . | . 4 . | . 1 9 


In [18]:
start = time.perf_counter()
def run_solver(board, max_steps = 200000, temp = 5, cooling_rate = 0.999):
    puzzle = copy.deepcopy(board)
    fixed_coordinates = find_fixed_cells(puzzle)
    swappable_cells = get_swappable_cells_block(fixed_coordinates)

    initialize_complete_board(puzzle, fixed_coordinates)
    current_energy = total_energy(puzzle)
    energy_history = []

    best_energy = current_energy

    current_step = 0

    for _ in range(max_steps):
        current_step += 1
        cell1, cell2 = choose_random_cells_random_block(swappable_cells)
        delta_energy = delta_energy_after_swap(puzzle, cell1, cell2)

        if not accept_move(delta_energy, temp):
            swap_cells(puzzle, cell1, cell2)
            energy_history.append(current_energy)
        else:
            current_energy += delta_energy
            energy_history.append(current_energy)
            best_energy = min(best_energy, current_energy)

        if current_energy == 0:
            #print('puzzle solved')
            #draw_board(puzzle)
            #return puzzle, current_energy, energy_history
            return True, current_step

        temp = update_temp(temp, cooling_rate)

    #return puzzle, current_energy, energy_history
    return False, current_step




# puzzle, energy, plot_energy = run_solver(board, 500000, 6, 0.99999)
#
# plt.plot(plot_energy)
# plt.xlabel("Iteration")
# plt.ylabel("Energy")
# plt.title("Energy during Simulated Annealing")
# plt.show()
# end = time.perf_counter()
# t = end - start
# print(energy)
# print(t)
solved, at_step = run_solver(board, 500000, 4.1195600493256554, 0.99)
solved, at_step




(False, 500000)

In [12]:
draw_board(board)

5 7 . | . 6 . | . . 3 
. 3 . | . . 5 | . 6 . 
6 . 1 | . . 7 | . . . 
---------------------
. 5 3 | . . . | . . 1 
. . . | . 8 . | . . . 
9 . . | . . . | 2 7 . 
---------------------
. . . | 8 . . | 4 . 2 
. 8 . | 1 . . | . 3 . 
2 . . | . 4 . | . 1 9 


In [13]:
energies_for_temperature = []
for _ in range(200):
    random_board = copy.deepcopy(board)
    fixed = find_fixed_cells(random_board)
    swappable_cells = get_swappable_cells_block(fixed)
    initialize_complete_board(random_board, fixed)
    this_energy = total_energy(random_board)
    energies_for_temperature.append(this_energy)

energies_for_temperature

[40,
 45,
 45,
 39,
 41,
 49,
 44,
 44,
 43,
 41,
 39,
 40,
 51,
 49,
 43,
 50,
 45,
 40,
 51,
 51,
 49,
 52,
 44,
 42,
 48,
 45,
 45,
 45,
 49,
 47,
 49,
 45,
 48,
 44,
 47,
 43,
 41,
 42,
 43,
 46,
 47,
 47,
 41,
 42,
 47,
 56,
 53,
 46,
 43,
 40,
 42,
 39,
 43,
 46,
 36,
 49,
 47,
 39,
 55,
 40,
 48,
 43,
 50,
 53,
 46,
 53,
 42,
 44,
 40,
 46,
 43,
 46,
 48,
 32,
 45,
 43,
 38,
 42,
 45,
 48,
 41,
 42,
 52,
 43,
 45,
 41,
 45,
 44,
 44,
 43,
 41,
 46,
 37,
 46,
 48,
 43,
 40,
 44,
 45,
 44,
 52,
 51,
 51,
 40,
 49,
 34,
 43,
 40,
 46,
 45,
 40,
 42,
 43,
 47,
 44,
 48,
 42,
 40,
 46,
 46,
 45,
 40,
 39,
 41,
 47,
 50,
 46,
 42,
 43,
 39,
 46,
 46,
 41,
 47,
 39,
 42,
 46,
 49,
 53,
 49,
 44,
 41,
 39,
 39,
 46,
 42,
 42,
 43,
 46,
 42,
 39,
 39,
 45,
 44,
 39,
 50,
 39,
 45,
 46,
 49,
 43,
 44,
 46,
 39,
 40,
 48,
 49,
 37,
 46,
 43,
 44,
 45,
 38,
 43,
 38,
 39,
 44,
 43,
 39,
 48,
 48,
 45,
 42,
 46,
 44,
 43,
 39,
 45,
 45,
 44,
 45,
 51,
 41,
 35,
 41,
 39,
 45,
 53,
 38,
 43]

In [16]:
st_dev_energy = np.std(energies_for_temperature)
print(st_dev_energy)

4.1195600493256554


In [ ]:
steps = [50000,100000,200000,500000,1000000]
temps = [x for x in range(1,11)]
coolings = [0.995, 0.999, 0.9995, 0.9999, 0.99995]
successful_parameters = []
all_parameters = []
count = 0
for step in steps:
    for temp in temps:
        for cooling in coolings:
            start = time.perf_counter()
            solved, at_step = run_solver(board,step,temp,cooling)
            end = time.perf_counter()
            t = end - start
            if solved:
                successful_parameters.append((step, temp, cooling, t, at_step))

            count += 1
            all_parameters.append((step, temp, cooling, t, solved))
            print(f'Iteration: {count}, step: {step}, temp: {temp}, cooling: {cooling}, time: {t:.2f}s, solved: {solved} at step: {at_step}')

successful_parameters

Iteration: 1, step: 50000, temp: 1, cooling: 0.995, time: 0.68s, solved: True at step: 12332
Iteration: 2, step: 50000, temp: 1, cooling: 0.999, time: 2.61s, solved: False at step: 50000
Iteration: 3, step: 50000, temp: 1, cooling: 0.9995, time: 2.56s, solved: False at step: 50000
Iteration: 4, step: 50000, temp: 1, cooling: 0.9999, time: 2.65s, solved: False at step: 50000
Iteration: 5, step: 50000, temp: 1, cooling: 0.99995, time: 1.28s, solved: True at step: 25696
Iteration: 6, step: 50000, temp: 2, cooling: 0.995, time: 2.70s, solved: False at step: 50000
Iteration: 7, step: 50000, temp: 2, cooling: 0.999, time: 3.29s, solved: False at step: 50000
Iteration: 8, step: 50000, temp: 2, cooling: 0.9995, time: 7.64s, solved: False at step: 50000
Iteration: 9, step: 50000, temp: 2, cooling: 0.9999, time: 7.25s, solved: False at step: 50000
Iteration: 10, step: 50000, temp: 2, cooling: 0.99995, time: 5.72s, solved: True at step: 36511
Iteration: 11, step: 50000, temp: 3, cooling: 0.995, t

In [ ]:
len(successful_parameters)

In [60]:
# Validation checks

def is_valid_row(board, row):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for cell in range(9):
        digits.add(board[row][cell])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_col(board, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()
    for cell in range(9):
        digits.add(board[cell][col])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_block(board, row, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            digits.add(board[r][c])

    if digits == desired_set:
        return True
    else:
        return False

def is_solved(board):
    errors = 0
    for row in range(9):
        is_valid = is_valid_row(board, row)
        if not is_valid:
            errors += 1
            print(f'Board not valid at row {row+1}')

    for col in range(9):
        is_valid = is_valid_col(board, col)
        if not is_valid:
            errors += 1
            print(f'Board not valid at col {col+1}')

    for row in range(0,9,3):
        for col in range(0,9,3):
            is_valid = is_valid_block(board, row, col)
            if not is_valid:
                errors += 1
                print(f'Board not valid at block ({row+1}, {col+1})')

    if errors == 0:
        print(f'Board is solved, great success')
    else:
        print(f'Board is obviously not solved :(')




In [61]:
is_solved(puzzle)

Board is solved, great success
